# N3DV Multi-View Frame Decoding Time Benchmark (OpenCV / ffmpeg-python)

Sibling of `n3dv_decode_timing.ipynb` (which uses PyAV). This notebook uses
**OpenCV** (`cv2.VideoCapture`) and **ffmpeg-python** (rawvideo pipe) instead.
Both backends **fuse demux + decode** and only expose a frame iterator, so —
as requested — there is **no demux/decode split**; only the combined
per-bundle decode time is measured.

Sweep axes:
1. **Decoder backend** — `opencv`, `ffmpeg-python`.
2. **Frame quality** — H.264 CRF / resolution variants.
3. **Camera count** — `1, 2, 4, 8, 16, 20`.
4. **Parallel mode** — `sync` (baseline), `thread`, `process`.
5. **Worker count** — fixed sweep + `"auto"` (= one worker per camera).

**Synchronized multi-view scenario.** A streaming 3DGS pipeline consumes one
*timestep* at a time: at timestep `t` it needs frame `t` from **every** camera
before fitting Gaussians. Workers decode assigned views; a **barrier** blocks
each bundle until all workers produced frame `t`, then `t+1` starts — like a
DataLoader collating one batch from `num_workers` before the next.

**Three phases:**

| Phase | What it covers |
|-------|----------------|
| `T_open`   | Open all streams (OpenCV) / spawn ffmpeg subprocesses + spin up workers |
| `T_first`  | Decode the **first** synchronized multi-view bundle (cold) |
| `T_steady` | Per-bundle latency **after warm-up** (median over steady timesteps) |

Decode backends live in [`mv_cv_ffmpeg_decode.py`](./mv_cv_ffmpeg_decode.py)
(a real module so `multiprocessing` workers import under fork **and** spawn).

> **Environment.** Run with the `3dgs` conda env
> (`/home/hheo/miniforge3/envs/3dgs/bin/python`). Deps installed there:
> `ffmpeg`, `opencv` (`cv2`), `ffmpeg-python`, `pandas`, `numpy`, `matplotlib`.

## Config

In [ ]:
from pathlib import Path

# --- Data location (parameterized; fill in when real data is ready) ---------
# N3DV scene dir: per-camera videos cam00.mp4 .. camNN.mp4 (DyNeRF release) OR
# cam00/cam01/... subdirs of extracted PNGs (QUEEN layout). Empty -> synthetic.
N3DV_SCENE_DIR = "/home/hheo/data/dynerf/n3dv/cut_roasted_beef/videos"

WORK_DIR = Path("./n3dv_decode_bench_cvff")

# --- Sweep axes ------------------------------------------------------------
# Decoder backends to compare. Both fuse demux+decode (no split).
DECODER_BACKENDS = ["opencv", "ffmpeg"]

QUALITY_VARIANTS = {
    "lossless":  dict(crf=0,  scale=1.0),
    "high_q18":  dict(crf=18, scale=1.0),
    "med_q28":   dict(crf=28, scale=1.0),
    "low_q38":   dict(crf=38, scale=1.0),
    "half_q23":  dict(crf=23, scale=0.5),
}

CAMERA_COUNTS = [1, 2, 4, 8, 16, 20]

# Total timesteps per measurement (1 timestep = 1 frame from EVERY camera).
NUM_FRAMES = 30

# Bundles excluded from T_steady (bundle 0 = T_first, reported separately).
WARMUP_BUNDLES = 3

# Repeat each measurement; report median.
NUM_REPEATS = 3

# Parallel modes: sync (baseline) / thread / process.
PARALLEL_MODES = ["sync", "thread", "process"]

# Worker-count sweep. Ints -> fixed pool; "auto" -> one worker per camera.
# Ignored for "sync" (always 1).
WORKER_COUNTS = ["auto", 2, 4, 8]

# process mode: ship pixels back to parent (realistic) vs just shape (pure
# decode throughput). sync/thread always see real frames.
PROCESS_RETURN_PIXELS = False

# --- Synthetic-data settings (only used when N3DV_SCENE_DIR is empty) -------
SYNTH_NUM_CAMERAS = 20
SYNTH_NUM_FRAMES = 90
SYNTH_RESOLUTION = (1352, 1014)  # N3DV native is 2704x2028; /2 for speed
SYNTH_FPS = 30

import os
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("WORK_DIR :", WORK_DIR.resolve())
print("Scene dir:", N3DV_SCENE_DIR or "(none -> synthetic)")
print(f"BACKENDS={DECODER_BACKENDS}  MODES={PARALLEL_MODES}  "
      f"WORKER_COUNTS={WORKER_COUNTS}  cpu_count={os.cpu_count()}")
print(f"NUM_FRAMES={NUM_FRAMES}  WARMUP_BUNDLES={WARMUP_BUNDLES}  "
      f"NUM_REPEATS={NUM_REPEATS}")
assert NUM_FRAMES > WARMUP_BUNDLES + 1, "Need >1 steady bundle after T_first + warm-up"

## Environment / backend availability

In [ ]:
import shutil, subprocess, importlib, time, json
import numpy as np
import pandas as pd

FFMPEG = shutil.which("ffmpeg")
FFPROBE = shutil.which("ffprobe")
print("ffmpeg :", FFMPEG)
print("ffprobe:", FFPROBE)

try:
    import cv2
    print("OpenCV :", cv2.__version__)
except Exception as e:
    raise RuntimeError("opencv-python is required") from e

try:
    import ffmpeg as ffmpeg_py
    assert hasattr(ffmpeg_py, "input") and hasattr(ffmpeg_py, "probe")
    print("ffmpeg-python: OK")
except Exception as e:
    raise RuntimeError("ffmpeg-python is required (pip install ffmpeg-python)") from e

assert FFMPEG, "ffmpeg binary required to build quality variants / synthetic data."
for b in DECODER_BACKENDS:
    assert b in ("opencv", "ffmpeg"), f"unknown backend {b!r}"

## Locate or synthesize source videos

`SOURCE_VIDEOS`: ordered per-camera mp4 paths. PNG-only scenes (QUEEN layout)
are encoded once into near-lossless mp4 so decoding can be timed uniformly.

In [ ]:
def run(cmd):
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def make_synthetic_videos():
    out_dir = WORK_DIR / "synthetic_source"
    out_dir.mkdir(parents=True, exist_ok=True)
    W, H = SYNTH_RESOLUTION
    paths = []
    for c in range(SYNTH_NUM_CAMERAS):
        p = out_dir / f"cam{c:02d}.mp4"
        paths.append(p)
        if p.exists():
            continue
        cmd = [
            FFMPEG, "-y", "-f", "lavfi",
            "-i", f"testsrc2=size={W}x{H}:rate={SYNTH_FPS}:duration={SYNTH_NUM_FRAMES/SYNTH_FPS:.4f}",
            "-vf", f"hue=h={c*17}",
            "-c:v", "libx264", "-preset", "medium", "-crf", "15",
            "-pix_fmt", "yuv420p", str(p),
        ]
        print("synth", p.name)
        run(cmd)
    return paths


def pngs_to_video(png_dir, out_path, fps=30):
    pngs = sorted(Path(png_dir).glob("*.png"))
    if not pngs:
        return None
    run([
        FFMPEG, "-y", "-framerate", str(fps),
        "-i", str(Path(png_dir) / "%04d.png"),
        "-c:v", "libx264", "-preset", "medium", "-crf", "12",
        "-pix_fmt", "yuv420p", str(out_path),
    ])
    return out_path


def locate_source_videos():
    if not N3DV_SCENE_DIR:
        return make_synthetic_videos()
    scene = Path(N3DV_SCENE_DIR)
    assert scene.is_dir(), f"N3DV_SCENE_DIR not found: {scene}"
    mp4s = sorted(scene.glob("cam*.mp4"))
    if mp4s:
        print(f"found {len(mp4s)} mp4 streams in {scene}")
        return mp4s
    cam_dirs = sorted(d for d in scene.glob("cam*") if d.is_dir())
    assert cam_dirs, f"No cam*.mp4 and no cam*/ dirs under {scene}"
    enc_dir = WORK_DIR / "encoded_from_png"
    enc_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for d in cam_dirs:
        png_dir = d / "images" if (d / "images").is_dir() else d
        out = enc_dir / f"{d.name}.mp4"
        if not out.exists():
            print("encode", d.name, "<-", png_dir)
            pngs_to_video(png_dir, out)
        if out.exists():
            paths.append(out)
    return paths


SOURCE_VIDEOS = locate_source_videos()
print(f"\n{len(SOURCE_VIDEOS)} source camera videos")
for p in SOURCE_VIDEOS[:3]:
    print("  ", p)

## Build quality variants

Re-encode every source stream into each `QUALITY_VARIANTS` setting once →
`WORK_DIR/variants/<quality>/cam<NN>.mp4`.

In [ ]:
def file_mb(p):
    return Path(p).stat().st_size / 1e6


VARIANT_DIR = WORK_DIR / "variants"
variant_videos = {}
variant_stats = []

for q, params in QUALITY_VARIANTS.items():
    out_dir = VARIANT_DIR / q
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for src in SOURCE_VIDEOS:
        out = out_dir / Path(src).name
        if not out.exists():
            vf = []
            if params["scale"] != 1.0:
                vf.append(f"scale=iw*{params['scale']}:ih*{params['scale']}")
            cmd = [FFMPEG, "-y", "-i", str(src)]
            if vf:
                cmd += ["-vf", ",".join(vf)]
            cmd += [
                "-c:v", "libx264", "-preset", "medium",
                "-crf", str(params["crf"]), "-pix_fmt", "yuv420p", str(out),
            ]
            run(cmd)
        paths.append(out)
    variant_videos[q] = paths
    total_mb = sum(file_mb(p) for p in paths)
    variant_stats.append(dict(quality=q, crf=params["crf"], scale=params["scale"],
                              n_cams=len(paths), total_MB=round(total_mb, 1),
                              MB_per_cam=round(total_mb / len(paths), 2)))
    print(f"{q:>10}: {len(paths)} cams, {total_mb:7.1f} MB total")

pd.DataFrame(variant_stats)

## Decoder backends (OpenCV / ffmpeg-python)

[`mv_cv_ffmpeg_decode.py`](./mv_cv_ffmpeg_decode.py) —
`run_decode(paths, num_frames, warmup, mode, backend, num_workers, ...)`:

- **`backend="opencv"`** — `cv2.VideoCapture`; one `cap.read()` per frame.
- **`backend="ffmpeg"`** — one ffmpeg subprocess per camera streaming rawvideo
  (rgb24) to a pipe; read `W*H*3` bytes per frame.
- **`mode`** — `sync` (1 thread), `thread`, `process`; same barrier-synced
  multi-view bundle protocol as the PyAV notebook.

Both backends fuse demux+decode → only the combined per-bundle time is timed
(no demux/decode split, by design).

In [ ]:
import importlib
import mv_cv_ffmpeg_decode as mvd
importlib.reload(mvd)

run_decode = mvd.run_decode
resolve_workers = mvd.resolve_workers

# Sanity probe (not timed): first 2 videos, all backend x mode combos.
_probe = SOURCE_VIDEOS[:2]
for _be in DECODER_BACKENDS:
    for _m in PARALLEL_MODES:
        r = run_decode(_probe, 6, 1, _m, backend=_be, num_workers="auto",
                        return_pixels=PROCESS_RETURN_PIXELS)
        print(f"{_be:>7} {_m:>8}: bundles={r['bundles']} "
              f"workers={r['n_workers']} open={1000*r['t_open']:.1f}ms "
              f"steady={1000*r['t_steady']:.2f}ms")

## Benchmark: backend × quality × cameras × mode × workers

Median across `NUM_REPEATS` of each phase. `sync` collapses the worker axis to
1; `"auto"` → `n_cams`; fixed ints clamped to `n_cams`; duplicate resolved
worker counts de-duped. No demux/decode split (fused backends).

In [ ]:
max_cams = len(SOURCE_VIDEOS)
cam_counts = sorted({min(c, max_cams) for c in CAMERA_COUNTS})
print("camera counts:", cam_counts, "(max available:", max_cams, ")")
print("backends:", DECODER_BACKENDS, " modes:", PARALLEL_MODES,
      " worker sweep:", WORKER_COUNTS, "\n")

rows = []
for backend in DECODER_BACKENDS:
    for q in QUALITY_VARIANTS:
        vids = variant_videos[q]
        for n_cams in cam_counts:
            subset = vids[:n_cams]
            for mode in PARALLEL_MODES:
                if mode == "sync":
                    worker_opts = [("sync", 1)]
                else:
                    seen, worker_opts = set(), []
                    for wc in WORKER_COUNTS:
                        nw = resolve_workers(wc, n_cams)
                        if nw in seen:
                            continue
                        seen.add(nw)
                        label = "auto" if wc == "auto" else str(wc)
                        worker_opts.append((label, nw))

                for wlabel, nw in worker_opts:
                    reps = [run_decode(subset, NUM_FRAMES, WARMUP_BUNDLES, mode,
                                        backend=backend, num_workers=nw,
                                        return_pixels=PROCESS_RETURN_PIXELS)
                            for _ in range(NUM_REPEATS)]
                    t_open = float(np.median([r["t_open"] for r in reps]))
                    t_first = float(np.median([r["t_first"] for r in reps]))
                    t_steady = float(np.median([r["t_steady"] for r in reps]))
                    steady_std = float(np.median([r["steady_std"] for r in reps]))
                    bundles = int(np.median([r["bundles"] for r in reps]))

                    rows.append(dict(
                        backend=backend,
                        quality=q,
                        crf=QUALITY_VARIANTS[q]["crf"],
                        scale=QUALITY_VARIANTS[q]["scale"],
                        n_cams=n_cams,
                        mode=mode,
                        workers=wlabel,
                        n_workers=nw,
                        bundles=bundles,
                        T_open_s=round(t_open, 4),
                        T_first_s=round(t_first, 4),
                        T_steady_s=round(t_steady, 4),
                        T_steady_ms=round(1000 * t_steady, 2),
                        steady_std_ms=round(1000 * steady_std, 2),
                        per_cam_steady_ms=round(1000 * t_steady / n_cams, 3),
                        stream_fps=round(1.0 / t_steady, 2) if t_steady > 0 else float("inf"),
                    ))
                    print(f"{backend:>7} | {q:>9} | {n_cams:>2}cam | {mode:>7} | "
                          f"w={wlabel:>4}({nw:>2}) | "
                          f"open {1000*t_open:7.1f} | first {1000*t_first:7.1f} | "
                          f"steady {1000*t_steady:7.2f} ms (~{1.0/t_steady:6.1f} fps)")

df = pd.DataFrame(rows)
csv_path = WORK_DIR / "decode_timing_cvff_results.csv"
df.to_csv(csv_path, index=False)
print("\nsaved ->", csv_path, " rows:", len(df))
df

## Results: tables

In [ ]:
# Speedup vs the sync baseline (same backend + quality + n_cams).
sync = (df[df["mode"] == "sync"]
        .set_index(["backend", "quality", "n_cams"])["T_steady_s"]
        .rename("sync_steady_s"))
df = df.join(sync, on=["backend", "quality", "n_cams"])
df["speedup_vs_sync"] = (df["sync_steady_s"] / df["T_steady_s"]).round(2)

df["config"] = df.apply(
    lambda r: f"{r['backend']}/sync" if r["mode"] == "sync"
    else f"{r['backend']}/{r['mode']}/w={r['workers']}", axis=1)

QUAL = "high_q18" if "high_q18" in QUALITY_VARIANTS else list(QUALITY_VARIANTS)[0]
qd = df[df.quality == QUAL]
print(f"Tables below are for quality = {QUAL!r}\n")

print("=== T_steady (ms/bundle) — rows=config, cols=n_cams ===")
display(qd.pivot_table(index="config", columns="n_cams", values="T_steady_ms").round(2))

print("\n=== backend head-to-head: T_steady (ms) at sync, by quality x n_cams ===")
sy = df[df["mode"] == "sync"]
display(sy.pivot_table(index=["backend", "quality"], columns="n_cams",
                       values="T_steady_ms").round(2))

print("\n=== speedup vs sync (×) ===")
display(qd.pivot_table(index="config", columns="n_cams", values="speedup_vs_sync").round(2))

print("\n=== T_open (ms) — stream open / ffmpeg spawn + worker spin-up ===")
display(qd.pivot_table(index="config", columns="n_cams", values="T_open_s")
          .mul(1000).round(1))

print("\n=== T_first (ms) — first synchronized bundle (cold) ===")
display(qd.pivot_table(index="config", columns="n_cams", values="T_first_s")
          .mul(1000).round(1))

print("\n=== sustainable streaming FPS (1 / T_steady) ===")
display(qd.pivot_table(index="config", columns="n_cams", values="stream_fps").round(1))

print("\n=== best config per (backend, quality, n_cams) by T_steady ===")
best = (df.loc[df.groupby(["backend", "quality", "n_cams"])["T_steady_s"].idxmin()]
          [["backend", "quality", "n_cams", "config", "T_steady_ms",
            "speedup_vs_sync", "stream_fps"]]
          .sort_values(["backend", "quality", "n_cams"]))
display(best)

## Results: plots

In [ ]:
import matplotlib.pyplot as plt

QUAL = "high_q18" if "high_q18" in QUALITY_VARIANTS else list(QUALITY_VARIANTS)[0]
qd = df[df.quality == QUAL]
configs = sorted(qd["config"].unique())

fig, axes = plt.subplots(1, 3, figsize=(19, 5))

# (1) T_steady vs camera count, one line per config
ax = axes[0]
for cfg in configs:
    s = qd[qd.config == cfg].sort_values("n_cams")
    ax.errorbar(s.n_cams, s.T_steady_ms, yerr=s.steady_std_ms,
                marker="o", capsize=3, label=cfg)
ax.set_xlabel("# cameras"); ax.set_ylabel("T_steady (ms / bundle)")
ax.set_title(f"Warm bundle latency  [{QUAL}]")
ax.legend(fontsize=7); ax.grid(alpha=.3)

# (2) backend head-to-head at sync across quality (max cameras)
ax = axes[1]
full = df[df.n_cams == df.n_cams.max()]
syf = full[full["mode"] == "sync"]
order = list(QUALITY_VARIANTS)
for be in DECODER_BACKENDS:
    s = syf[syf.backend == be].set_index("quality").reindex(order).reset_index()
    ax.plot(s.quality, s.T_steady_ms, marker="o", label=f"{be}/sync")
ax.set_xlabel("quality"); ax.set_ylabel("T_steady (ms / bundle)")
ax.set_title(f"OpenCV vs ffmpeg-python (sync) @ {df.n_cams.max()} cams")
ax.tick_params(axis="x", rotation=30); ax.legend(fontsize=8); ax.grid(alpha=.3)

# (3) T_open vs camera count per config (ffmpeg spawn cost shows here)
ax = axes[2]
for cfg in configs:
    s = qd[qd.config == cfg].sort_values("n_cams")
    ax.plot(s.n_cams, s.T_open_s * 1000, marker="s", label=cfg)
ax.set_xlabel("# cameras"); ax.set_ylabel("T_open (ms)")
ax.set_title(f"Open + spawn + worker spin-up  [{QUAL}]")
ax.legend(fontsize=7); ax.grid(alpha=.3)

plt.tight_layout()
plt.savefig(WORK_DIR / "decode_timing_cvff.png", dpi=120, bbox_inches="tight")
plt.show()

## Streaming-3DGS interpretation

In [ ]:
TARGET_TRAIN_FPS = 1.0  # timesteps/sec the 3DGS fitter can sustain
budget_s = 1.0 / TARGET_TRAIN_FPS
print(f"Decode budget per timestep @ {TARGET_TRAIN_FPS} train-fps = {budget_s:.3f}s\n")

g = df.loc[df.groupby(["backend", "quality", "n_cams"])["T_steady_s"].idxmin()].copy()
g["fits_budget"] = np.where(g["T_steady_s"] <= budget_s, "ok", "<<BOTTLENECK")
print("=== best config per (backend, quality, n_cams) vs budget ===")
display(g[["backend", "quality", "n_cams", "config", "T_steady_ms",
           "speedup_vs_sync", "stream_fps", "fits_budget"]]
        .sort_values(["backend", "quality", "n_cams"]).reset_index(drop=True))

fast = df.loc[df.T_steady_s.idxmin()]
slow = df.loc[df.T_steady_s.idxmax()]
print(f"\nFastest overall : {fast['config']} | {fast['quality']} | "
      f"{fast['n_cams']} cams -> {fast['T_steady_ms']} ms/bundle "
      f"({fast['stream_fps']} fps)")
print(f"Slowest overall : {slow['config']} | {slow['quality']} | "
      f"{slow['n_cams']} cams -> {slow['T_steady_ms']} ms/bundle "
      f"({slow['stream_fps']} fps)")

print("\n=== mean T_steady (ms) by backend (all configs) ===")
display(df.groupby("backend")["T_steady_ms"].agg(["mean", "min", "max"]).round(2))

print("\n=== mean T_open / T_first (ms) by backend — startup cost ===")
tmp = df.copy()
tmp["T_open_ms"] = 1000 * tmp["T_open_s"]
tmp["T_first_ms"] = 1000 * tmp["T_first_s"]
display(tmp.groupby("backend")[["T_open_ms", "T_first_ms"]].mean().round(1))